# Person 4 (Neha): second-reader helper
Runs the **mechanical** checks on a task-type sheet (blank cells, spacing, spelling variants, numbers, length, overlap with the 232 existing pairs).
It does **not** replace reading every pair for meaning, difficulty and natural Hinglish. Use it to find the boring problems fast, then read.
Exploratory tool: nothing here changes any frozen file.

In [ ]:
from google.colab import files
up = files.upload()   # upload: the sheet to check, confirmatory_v1.csv, benchmark_v2.xlsx

In [ ]:
import pandas as pd, re, collections

REQUIRED = ['pair_id','task_type','difficulty','source','source_id','written_by','checked_by',
            'check_status','check_note','english','hinglish']
VARIANTS = {   # groups of spellings that should be unified
 'in':        ['mein','me','main'],
 'we/us':     ['hamein','humein','hume','hamen'],
 'why':       ['kyun','kyu','kyon','kyunki'],
 'then':      ['toh','to'],
 'on':        ['par','pe','pr'],
 'no/not':    ['nahi','nahin','nai'],
 'for':       ['liye','lie'],
 'how':       ['kaise','kese','kaisey'],
}
HINDI_WORDS = set('ki ka ke ko hai hain ho kar karo kare se par me mein aur ya jo kya kyun kaise ek do teen nahi toh bhi hota hote hoti'.split())

def words(t): return set(re.findall(r"[a-z0-9_]+", str(t).lower()))
def jacc(a,b): 
    a,b=words(a),words(b); return len(a&b)/max(1,len(a|b))
def toks(t): return re.findall(r"[a-zA-Z0-9_']+", str(t).lower())

def check(sheet, existing_en, ref_hinglish=None, thr=0.5):
    d = sheet.copy().reset_index(drop=True)
    out = {}
    out['schema_ok'] = [c for c in REQUIRED if c not in d.columns]
    d['en_words'] = d.english.fillna('').str.split().str.len()
    d['hi_words'] = d.hinglish.fillna('').str.split().str.len()
    flags = [[] for _ in range(len(d))]
    for i,r in d.iterrows():
        e,h = str(r.english), str(r.hinglish)
        if e.strip()=='' or h.strip()=='' or e=='nan' or h=='nan': flags[i].append('blank text')
        if '  ' in e or '  ' in h or e!=e.strip() or h!=h.strip(): flags[i].append('extra spaces')
        if e.strip().lower()==h.strip().lower(): flags[i].append('hinglish identical to english')
        if not (set(toks(h)) & HINDI_WORDS): flags[i].append('no Hindi function words')
        if re.search(r'router|hinglish|code-?mix|this study|routing', e+' '+h, re.I): flags[i].append('mentions the study')
        nums_e = sorted(re.findall(r'\d+(?:\.\d+)?', e)); nums_h = sorted(re.findall(r'\d+(?:\.\d+)?', h))
        if nums_e != nums_h: flags[i].append(f'numbers differ {nums_e} vs {nums_h}')
        if r.en_words>0 and (r.hi_words/r.en_words>1.35 or r.hi_words/r.en_words<0.8): flags[i].append(f'length ratio {r.hi_words/r.en_words:.2f}')
        # overlap with existing 232 and within sheet
        best = max((jacc(e,x) for x in existing_en), default=0)
        if best>=thr: flags[i].append(f'close to an existing pair (overlap {best:.2f})')
        for j in range(i):
            if jacc(e, d.english[j])>=0.6: flags[i].append(f'near-duplicate of row {j+2}')
    d['flag_notes'] = ['; '.join(f) for f in flags]
    # spelling variant counts
    alltxt = ' '.join(d.hinglish.fillna('').str.lower())
    vc = {}
    for g,vs in VARIANTS.items():
        c = {v: len(re.findall(r'\b'+v+r'\b', alltxt)) for v in vs}
        c = {k:v for k,v in c.items() if v}
        if len(c)>1 or (c and ref_hinglish is not None): vc[g]=c
    out['variants']=vc
    out['table']=d
    return out

In [ ]:
import pandas as pd, re
SHEET = 'thirdset_openended_v3.csv'      # <-- change to the sheet you are checking
conf  = pd.read_csv('confirmatory_v1.csv')
pilot = pd.read_excel('benchmark_v2.xlsx', sheet_name='Benchmark')
existing = list(conf['english']) + list(pilot['english'])

d = pd.read_csv(SHEET)
o = check(d, existing, ref_hinglish=True)
t = o['table']
print('missing columns:', o['schema_ok'])
print(t.groupby('difficulty')[['en_words','hi_words']].agg(['min','mean','max']).round(1))
print('\nspelling variants (more than one form of the same word means: unify):')
for g, c in o['variants'].items(): print('  ', g, c)
print('\nrows with flags (sheet row = index + 2):')
t[t['flag_notes'] != ''][['difficulty','english','flag_notes']]

In [ ]:
# How the existing 152 pairs spell things, per task type, so the team can pick ONE standard
for tt, g in conf.groupby('task_type'):
    tx = ' '.join(g['hinglish'].str.lower())
    print(tt, {w: len(re.findall(r'\b'+w+r'\b', tx)) for w in ['mein','me','toh','to','hamein','humein']})

In [ ]:
t.to_csv('review_report.csv', index=False)
files.download('review_report.csv')